# 1.3 Preprocessing — Full-Stack Test (500 planes)

This notebook re-runs **Step 1: Preprocessing** (background removal + PSF-centered
Richardson-Lucy deconvolution) on the **entire `cnx43.tif` stack** (500 planes),
instead of the small hand-picked subset used in `1_1_preprocessing.ipynb`.

QC output is driven by `zone_samples.csv`, which records one representative
`sample_plane` per tissue `zone_label` (plus a `sample_plane_intensity` reference
value used as a sanity check against freshly computed intensities).

**Note on runtime:** running rolling-ball background subtraction and Richardson-Lucy
deconvolution across 500 planes is substantially slower than the old 10-plane test.
Richardson-Lucy uses `method='gpu'` and will print a warning and fall back to CPU if
no GPU is available — expect a long wall-clock time in that case.


In [ ]:
import sys
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import io
import stackview
import RedLionfishDeconv as rl

BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR / 'src'))

from src.preprocessing import (
    remove_background_rolling_ball_3d,
    brightest_slice,
    center_psf,
    normalize_psf,
)


## 0. Load zone samples

Adjust `zone_samples_path` below if your file lives somewhere else.

In [ ]:
zone_samples_path = BASE_DIR / 'data' / 'zone_samples.csv'
zone_samples = pd.read_csv(zone_samples_path)
print(f"Loaded {len(zone_samples)} zones from {zone_samples_path}")
zone_samples


## 1. Load the full image stack

In [ ]:
cnx_path = BASE_DIR / 'data' / 'raw' / 'corrected_images' / 'cnx43.tif'
cnx_img = io.imread(cnx_path)
print(f"Loaded stack: shape={cnx_img.shape}, dtype={cnx_img.dtype}")

# remove black border on top (matches convention in 1_1_preprocessing.ipynb)
cnx_img = cnx_img[:, 15:, :]
print(f"After border crop: shape={cnx_img.shape}")

n_planes = cnx_img.shape[0]
if n_planes != 500:
    print(f"WARNING: expected 500 planes, got {n_planes}")

out_of_range = zone_samples[~zone_samples['sample_plane'].between(0, n_planes - 1)]
if len(out_of_range):
    print("WARNING: sample planes out of range for this stack:")
    print(out_of_range)
else:
    print("All zone_samples.sample_plane values are within the stack range.")


## 2. QC helpers

In [ ]:
def plot_zone_grid(stack, zone_df, title, cmap='gray', ncols=4):
    """One subplot per zone, showing that zone's sample_plane."""
    n = len(zone_df)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
    for i, (_, row) in enumerate(zone_df.iterrows()):
        ax = axes[i // ncols, i % ncols]
        plane = int(row['sample_plane'])
        ax.imshow(stack[plane], cmap=cmap)
        ax.set_title(f"{row['zone_label']} (plane {plane})")
        ax.axis('off')
    for j in range(n, nrows * ncols):
        axes[j // ncols, j % ncols].axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def zone_intensity_table(stack, zone_df, column_name):
    """Mean intensity at each zone's sample_plane."""
    values = [stack[int(row['sample_plane'])].mean() for _, row in zone_df.iterrows()]
    return pd.DataFrame({
        'zone_number': zone_df['zone_number'].values,
        column_name: values,
    })


## 3. Raw-stack QC

In [ ]:
plot_zone_grid(cnx_img, zone_samples, title="Raw stack — sample planes per zone")

raw_intensity = zone_intensity_table(cnx_img, zone_samples, 'computed_intensity_raw')
raw_comparison = zone_samples[['zone_number', 'zone_label', 'sample_plane', 'sample_plane_intensity']].merge(
    raw_intensity, on='zone_number'
)
raw_comparison['diff'] = raw_comparison['computed_intensity_raw'] - raw_comparison['sample_plane_intensity']
raw_comparison['pct_diff'] = 100 * raw_comparison['diff'] / raw_comparison['sample_plane_intensity']
print("Recorded vs. computed intensity at each zone's sample plane (raw stack):")
raw_comparison


## 4. Step 1 — Background removal (rolling ball)

In [ ]:
%time background_removed = remove_background_rolling_ball_3d(cnx_img, radius=10, n_jobs=-1)

bg_out_path = BASE_DIR / 'data' / 'preprocessed' / 'cnx43_full_stack_background_removed.tif'
bg_out_path.parent.mkdir(parents=True, exist_ok=True)
io.imsave(bg_out_path, background_removed)
print(f"Saved background-removed stack to {bg_out_path}")


## 5. QC of background removal

In [ ]:
plot_zone_grid(background_removed, zone_samples, title="Background-removed stack — sample planes per zone")

bg_intensity = zone_intensity_table(background_removed, zone_samples, 'computed_intensity_bg_removed')
bg_comparison = raw_comparison.merge(bg_intensity, on='zone_number')
bg_comparison['pct_reduction'] = 100 * (
    bg_comparison['computed_intensity_raw'] - bg_comparison['computed_intensity_bg_removed']
) / bg_comparison['computed_intensity_raw']
print("Per-zone intensity before/after background removal:")
bg_comparison[['zone_label', 'sample_plane', 'computed_intensity_raw', 'computed_intensity_bg_removed', 'pct_reduction']]


In [ ]:
plt.hist(background_removed.ravel(), bins=100)
plt.yscale('log')
plt.xlabel('pixel value')
plt.ylabel('count (log)')
plt.title('Background-removed stack — full intensity histogram')
plt.show()
print(f"min={background_removed.min()}, max={background_removed.max()}")


## 6. Step 2.1 — PSF centering

In [ ]:
psf_path = BASE_DIR / 'data' / 'raw' / 'experimental_PSF' / 'PSF_488nm_cnx.h5'
psf_name = 'PSF_488'

with h5py.File(psf_path, "r") as f:
    psf = f[f'{psf_name}/ImageData/Image'][:]
    print("shape of PSF as stored:", psf.shape)

# collapse leading singleton/channel dims down to a plain 3D (z, y, x) stack
psf_3d = np.squeeze(psf)
while psf_3d.ndim > 3:
    psf_3d = psf_3d[0]
print("PSF used for centering, shape:", psf_3d.shape)

brightest = brightest_slice(psf_3d)
print("Brightest PSF slice index:", brightest)

psf_centered = center_psf(psf_3d, brightest)
psf_normalized = normalize_psf(psf_centered)
print("Normalized PSF sum (should be 1):", psf_normalized.sum())

stackview.imshow(psf_normalized[psf_normalized.shape[0] // 2], colormap='gray', title="Centered PSF — middle slice")

psf_out_path = BASE_DIR / 'data' / 'raw' / 'experimental_PSF' / f"{psf_name}_centered.tiff"
io.imsave(psf_out_path, psf_normalized)
print(f"Saved centered PSF to {psf_out_path}")


## 7. Step 2.2 — Richardson-Lucy deconvolution

In [ ]:
image_float32 = background_removed.astype(np.float32)

%time deconvolved = rl.doRLDeconvolutionFromNpArrays(image_float32, psf_normalized[16:], niter=7, method='gpu')  # falls back to CPU if no GPU available

deconv_out_path = BASE_DIR / 'data' / 'preprocessed' / 'cnx43_full_stack_preprocessed.tif'
io.imsave(deconv_out_path, deconvolved)
print(f"Saved deconvolved full-stack image to {deconv_out_path}")


## 8. Final QC — raw vs. background-removed vs. deconvolved

In [ ]:
def plot_zone_stage_grid(stages, zone_df, cmap='gray'):
    """Grid of (zone rows) x (pipeline stage columns) at each zone's sample_plane."""
    n_zones = len(zone_df)
    n_stages = len(stages)
    fig, axes = plt.subplots(n_zones, n_stages, figsize=(4 * n_stages, 4 * n_zones), squeeze=False)
    for i, (_, row) in enumerate(zone_df.iterrows()):
        plane = int(row['sample_plane'])
        for j, (stage_name, stage_stack) in enumerate(stages.items()):
            ax = axes[i, j]
            ax.imshow(stage_stack[plane], cmap=cmap)
            if i == 0:
                ax.set_title(stage_name)
            if j == 0:
                ax.set_ylabel(f"{row['zone_label']}\n(plane {plane})", rotation=0, ha='right', va='center')
            ax.set_xticks([])
            ax.set_yticks([])
    plt.tight_layout()
    plt.show()

stages = {
    'raw': cnx_img,
    'background_removed': background_removed,
    'deconvolved': deconvolved,
}
plot_zone_stage_grid(stages, zone_samples)


In [ ]:
deconv_intensity = zone_intensity_table(deconvolved, zone_samples, 'computed_intensity_deconvolved')
final_summary = bg_comparison.merge(deconv_intensity, on='zone_number')
final_summary = final_summary[[
    'zone_number', 'zone_label', 'sample_plane', 'sample_plane_intensity',
    'computed_intensity_raw', 'computed_intensity_bg_removed', 'computed_intensity_deconvolved',
]]
print("Final per-zone intensity summary across all preprocessing stages:")
final_summary


## Summary

Wall-clock timing for the two heavy steps is printed above by the `%time` magic on
each cell (background removal in Section 4, deconvolution in Section 7). The table in
Section 8 gives the recorded vs. computed intensity at every stage for each zone —
large `pct_diff`/unexpected jumps there are the main signal to check before trusting
this full-stack run.